# gpt2 finetuned on penn tree bank

In [1]:
!pip install --quiet transformers datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 8.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2024.12.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cuda-

In [2]:
from google.colab import drive
drive.mount('/content/drive')

%cd drive/MyDrive/ColabNotebooks/NLP_HW_finetuning

MessageError: Error: credential propagation was unsuccessful

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
from datasets import load_dataset

In [ ]:
# 1. Model and tokenizer initialization
model_name = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
# for models like gpt-2 that don't have a pad token by default
tokenizer.pad_token = tokenizer.eos_token

# for other models without pad token
# tokenizer.add_special_tokens({'pad_token': '[PAD]'})

# 2. Load model to device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

# 3. Load Dataset
dataset = load_dataset("ptb_text_only")

train_data = dataset["train"]
val_data = dataset['validation']

# 4. Tokenization Function
def tokenize_function(example):
  result = tokenizer(
                  example["sentence"],
                  truncation=True,
                  padding="max_length", # could also use ’longest’
                  max_length=512, # set a reasonable max length
                  )
  result["labels"] = result["input_ids"].copy()
  return result


# 5. Apply Tokenization and REmove Original 'text' Column
tokenized_train = train_data.map(tokenize_function, batched=True, remove_columns=['sentence'])
tokenized_val = val_data.map(tokenize_function, batched=True, remove_columns=['sentence'])

# Remove columns other than input_ids/attention_mask
tokenized_train.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
tokenized_val.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
# tokenized_train = tokenized_train.with_format("torch", device=device)
# tokenized_val = tokenized_val.with_format("torch", device=device)
# 7. Training Arguments
training_args = TrainingArguments(
                            output_dir="./models/gpt2+ptb",
                            eval_strategy="epoch",
                            save_strategy="epoch",
                            num_train_epochs=1, # Increase for better results
                            per_device_train_batch_size=8, # Adjust based on GPU memory
                            per_device_eval_batch_size=8,
                            save_steps=500,
                            logging_steps=100,
                            load_best_model_at_end=True,
                            remove_unused_columns=False,
                            push_to_hub=False
                            )

# 8. Trainer Definition
trainer = Trainer(
              model=model,
              args=training_args,
              train_dataset=tokenized_train,
              eval_dataset=tokenized_val,
              tokenizer=tokenizer # tokenizer is important
              )

# 9. Training
trainer.train()

prompt = "The stock market reacted negatively to the news that'"
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.cuda()
# Generate text
output_ids = model.generate(
                          input_ids,
                          max_length=512,
                          num_beams=5,
                          no_repeat_ngram_size=2,
                          early_stopping=True
                          )
generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
print("Prompt:", prompt)
print("Generated␣text:\n", generated_text)

model.save_pretrained("./models/gpt2-ptb")
tokenizer.save_pretrained("./models/gpt2-ptb")